# 01 · Legacy spreadsheets profile (F11, US-011)

First read of the three original legacy files. This notebook only **calls** the tested
functions in `ml/legacy_profile/`; every rule lives in the package and its tests.

- Inputs (ignored by Git): `ml/data/raw/Clientes - Comercial.xlsx`,
  `ml/data/raw/Pedidos 2025-2026 - Comercial.xlsx`, `ml/data/raw/titulos.csv`, and the
  ERP export in `ml/data/raw/erp/` (`ml/scripts/export_erp.py`).
- Output: the tables used in `docs/data/legacy-spreadsheets-profile.md`, also written to
  `ml/data/raw/profile-tables.md` (ignored by Git).
- The original files are never edited. **Commit this notebook with outputs cleared**:
  outputs could contain client data.

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd

ML_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd() / 'ml'
sys.path.insert(0, str(ML_DIR))

from legacy_profile.anomalies import (blank_rows, cancelled_rows, invalid_quantities,
    reversed_rows, suspicious_quantities, total_rows, unknown_statuses)
from legacy_profile.duplicates import exact_duplicates, key_duplicates
from legacy_profile.erp_match import load_erp_export, match_codes, october_overlap
from legacy_profile.findings import rows_classified_as, rows_not_in
from legacy_profile.formats import (classify_date, classify_installment, classify_money,
    format_counts, normalize_installment, status_variants, structure_summary)
from legacy_profile.readers import read_table
from legacy_profile.references import classify_code, missing_references, reference_counts
from legacy_profile.report import build_file_report, render_markdown
from legacy_profile.text import is_blank

RAW = ML_DIR / 'data' / 'raw'
CUSTOMERS, ORDERS, RECEIVABLES = (
    'Clientes - Comercial.xlsx', 'Pedidos 2025-2026 - Comercial.xlsx', 'titulos.csv')
HEADER_ROW = {CUSTOMERS: 3, ORDERS: 4, RECEIVABLES: 1}  # found in the first read

## ERP export (reference for RN-09)

In [ ]:
erp = load_erp_export(RAW / 'erp')
assert erp is not None, 'Run ml/scripts/export_erp.py first'
erp_exported_at = datetime.fromtimestamp((RAW / 'erp' / 'clientes.json').stat().st_mtime)
ERP_CUSTOMERS = {c['codigo'] for c in erp['clientes']}
ERP_PRODUCTS = {p['codigo'] for p in erp['produtos']}
ERP_SELLERS = {c['vendedor'] for c in erp['clientes']}
ERP_RECEIVABLE_NUMBERS = [t['numero'] for t in erp['titulos']]
CUSTOMER_STATUSES = ['ATIVO', 'BLOQUEADO', 'INATIVO']
RECEIVABLE_STATUSES = ['PAGO', 'EM_ABERTO', 'VENCIDO']
print(f'Exported {erp_exported_at:%Y-%m-%d %H:%M}:',
      {name: len(rows) for name, rows in erp.items()}, '| sellers:', sorted(ERP_SELLERS))

## Read the original files as text

In [ ]:
frames = {name: read_table(RAW / name, header_row=HEADER_ROW[name])
          for name in (CUSTOMERS, ORDERS, RECEIVABLES)}
customers, orders, receivables = frames[CUSTOMERS], frames[ORDERS], frames[RECEIVABLES]
structure = {name: structure_summary(frame) for name, frame in frames.items()}
for name, summary in structure.items():
    print(name, summary)

## Helpers for this notebook

Small selections only (which rows to look at); the detection itself is in the package.
Blank rows are left out of duplicate checks, otherwise every empty separator would be
counted as a duplicate of the previous one.

In [ ]:
def code_rows(series, prefix, digits, classes):
    return rows_classified_as(series, lambda v: classify_code(v, prefix, digits), classes)

def codes_not_in_erp(series, prefix, digits, known):
    standard = code_rows(series, prefix, digits, {'standard'})
    return rows_not_in(series.loc[standard], known)

def without(frame, rows):
    return frame.drop(index=rows)

def with_value(frame, column):
    return frame[~frame[column].map(is_blank).astype(bool)]

def union(*indexes):
    result = indexes[0]
    for index in indexes[1:]:
        result = result.union(index, sort=False)
    return result

## Customers (`Clientes - Comercial.xlsx`)

In [ ]:
c = customers
c_blank = blank_rows(c)
# a customer row is identified by its code or its company name
c_total = total_rows(c, ['Cód.', 'Razão Social'], ['Limite'])
c_data = without(c, union(c_blank, c_total))
c_findings = {
    'blank_row': c_blank,
    'total_row': c_total,
    'exact_duplicate': exact_duplicates(c_data),
    'key_duplicate': key_duplicates(with_value(c_data, 'Cód.'), ['Cód.']),
    'missing_reference': missing_references(c_data, 'Cód.'),
    'off_pattern_code': code_rows(c_data['Cód.'], 'C', 4, {'off_pattern'}),
    'name_reference': code_rows(c_data['Cód.'], 'C', 4, {'name'}),
    'not_in_erp': codes_not_in_erp(c_data['Cód.'], 'C', 4, ERP_CUSTOMERS),
    'unmatched_seller': rows_not_in(c_data['Vendedor'], ERP_SELLERS),
    'unknown_status': unknown_statuses(c_data['Situação'], CUSTOMER_STATUSES),
    'unrecognized_money': rows_classified_as(c_data['Limite'], classify_money, {'unrecognized'}),
    'unrecognized_date': rows_classified_as(c_data['Últ. atualização'], classify_date, {'unrecognized'}),
}
c_formats = {
    'Limite (money)': format_counts(c['Limite'], classify_money),
    'Últ. atualização (date)': format_counts(c['Últ. atualização'], classify_date),
    'Cód. (code)': reference_counts(c['Cód.'], 'C', 4),
}
c_variants = {'Situação': status_variants(c['Situação']), 'Vendedor': status_variants(c['Vendedor'])}
{k: len(v) for k, v in c_findings.items()}

## Orders (`Pedidos 2025-2026 - Comercial.xlsx`)

In [ ]:
p = orders
p_blank = blank_rows(p)
p_total = total_rows(p, ['Nº Pedido', 'Cliente', 'Produto'], ['Qtd', 'Preço Unit.', 'Total'])
p_data = without(p, union(p_blank, p_total))
p_findings = {
    'blank_row': p_blank,
    'total_row': p_total,
    'exact_duplicate': exact_duplicates(p_data),
    'key_duplicate': key_duplicates(with_value(p_data, 'Produto'), ['Nº Pedido', 'Produto']),
    'missing_reference': union(missing_references(p_data, 'Cliente'), missing_references(p_data, 'Produto')),
    'off_pattern_code': union(code_rows(p_data['Cliente'], 'C', 4, {'off_pattern'}),
                              code_rows(p_data['Produto'], 'P', 5, {'off_pattern'})),
    'name_reference': union(code_rows(p_data['Cliente'], 'C', 4, {'name'}),
                            code_rows(p_data['Produto'], 'P', 5, {'name'})),
    'not_in_erp': union(codes_not_in_erp(p_data['Cliente'], 'C', 4, ERP_CUSTOMERS),
                        codes_not_in_erp(p_data['Produto'], 'P', 5, ERP_PRODUCTS)),
    'unmatched_seller': rows_not_in(p_data['Vendedor'], ERP_SELLERS),
    'invalid_quantity': invalid_quantities(p_data['Qtd']),
    # rows without a product code are a missing reference, not a product of their own
    'suspicious_quantity': suspicious_quantities(with_value(p_data, 'Produto'), 'Produto', 'Qtd'),
    'unrecognized_date': rows_classified_as(p_data['Data'], classify_date, {'unrecognized', 'empty'}),
    'unrecognized_money': rows_classified_as(p_data['Preço Unit.'], classify_money, {'unrecognized', 'empty'}),
    'cancelled_order': cancelled_rows(p_data['Status']),
}
p_formats = {
    'Data (date)': format_counts(p['Data'], classify_date),
    'Preço Unit. (money)': format_counts(p['Preço Unit.'], classify_money),
    'Cliente (code)': reference_counts(p['Cliente'], 'C', 4),
    'Produto (code)': reference_counts(p['Produto'], 'P', 5),
}
p_variants = {'Status': status_variants(p['Status']), 'Vendedor': status_variants(p['Vendedor'])}
p_total_filled = int((~p['Total'].map(is_blank)).sum())
p_invalid_values = p.loc[p_findings['invalid_quantity'], 'Qtd'].value_counts()
print('Total column filled in', p_total_filled, 'of', len(p), 'rows')
{k: len(v) for k, v in p_findings.items()}

## Receivables (`titulos.csv`)

In [ ]:
t = receivables
DATE_COLUMNS = ['data_emissao', 'data_vencimento', 'data_pagamento']
t_blank = blank_rows(t)
t_total = total_rows(t, ['id_titulo', 'id_cliente', 'num_pedido'], ['valor'])
t_data = without(t, union(t_blank, t_total))
t_reversed = reversed_rows(t_data['status'])
t_findings = {
    'blank_row': t_blank,
    'total_row': t_total,
    'exact_duplicate': exact_duplicates(t_data),
    'key_duplicate': key_duplicates(t_data, ['id_titulo', 'parcela'], {'parcela': normalize_installment}),
    'missing_reference': missing_references(t_data, 'id_cliente'),
    'off_pattern_code': code_rows(t_data['id_cliente'], 'C', 4, {'off_pattern'}),
    'name_reference': code_rows(t_data['id_cliente'], 'C', 4, {'name'}),
    'not_in_erp': codes_not_in_erp(t_data['id_cliente'], 'C', 4, ERP_CUSTOMERS),
    # payment date may be empty (not paid yet); issue and due dates may not
    'unrecognized_date': union(
        rows_classified_as(t_data['data_emissao'], classify_date, {'unrecognized', 'empty'}),
        rows_classified_as(t_data['data_vencimento'], classify_date, {'unrecognized', 'empty'}),
        rows_classified_as(t_data['data_pagamento'], classify_date, {'unrecognized'})),
    'unrecognized_money': rows_classified_as(t_data['valor'], classify_money, {'unrecognized', 'empty'}),
    'reversed_receivable': t_reversed,
    'unknown_status': unknown_statuses(t_data['status'], RECEIVABLE_STATUSES).difference(t_reversed, sort=False),
}
t_formats = {f'{col} (date)': format_counts(t[col], classify_date) for col in DATE_COLUMNS}
t_formats['valor (money)'] = format_counts(t['valor'], classify_money)
t_formats['parcela (installment)'] = format_counts(t['parcela'], classify_installment)
t_formats['id_cliente (code)'] = reference_counts(t['id_cliente'], 'C', 4)
t_variants = {'status': status_variants(t['status'])}
t_negative = int(t['valor'].str.strip().str.startswith('-').sum())
print('Negative amounts:', t_negative)
{k: len(v) for k, v in t_findings.items()}

## ERP cross-check

In [ ]:
def distinct_standard(series, prefix, digits):
    return series.loc[code_rows(series, prefix, digits, {'standard'})]

legacy_customer_codes = pd.concat([
    distinct_standard(customers['Cód.'], 'C', 4),
    distinct_standard(orders['Cliente'], 'C', 4),
    distinct_standard(receivables['id_cliente'], 'C', 4),
], ignore_index=True)
cross_check = {
    'customer codes (all files)': match_codes(legacy_customer_codes, ERP_CUSTOMERS),
    'product codes (orders)': match_codes(distinct_standard(orders['Produto'], 'P', 5), ERP_PRODUCTS),
    'October 2026 receivables': october_overlap(receivables, 'id_titulo', 'data_emissao', ERP_RECEIVABLE_NUMBERS),
}
cross_check

## Evidence for the proposed rules (exploratory)

How many flagged rows could be corrected safely: exactly one ERP record matches.
This is evidence for the sprint 2 load, not a correction; nothing is written.

In [ ]:
import re
from collections import Counter
from legacy_profile.text import normalize_text

ERP_BY_NAME = Counter(normalize_text(x['razaoSocial']) for x in erp['clientes'])
ERP_BY_DESCRIPTION = Counter(normalize_text(x['descricao']) for x in erp['produtos'])

def unique_match(values, index):
    return Counter('exactly one' if index[normalize_text(v)] == 1 else
                   'none' if index[normalize_text(v)] == 0 else 'several' for v in values)

def quantity_shape(value):
    v = value.strip()
    if re.fullmatch(r'\d+,0+', v): return "whole number written with ',0'"
    if re.fullmatch(r'\d+\s*[A-Za-z]+', v): return "number followed by a unit"
    return 'other'

off_codes = receivables.loc[t_findings['off_pattern_code'], 'id_cliente']
padded = off_codes.map(lambda v: 'C' + re.sub(r'\D', '', v).zfill(4))
order_customers = orders.groupby('Nº Pedido')['Cliente'].agg(lambda s: len(set(s)))
no_customer = receivables.loc[t_findings['missing_reference'], 'num_pedido']
cancelled_numbers = set(orders.loc[p_findings['cancelled_order'], 'Nº Pedido'])
evidence = {
    'receivables off-pattern code, zero-padded code exists in ERP':
        f"{int(padded.isin(ERP_CUSTOMERS).sum())} of {len(off_codes)}"
        f" ({int(off_codes.map(lambda v: len(re.sub(r'[^0-9]', '', v)) == 3).sum())} have only 3 digits)",
    'orders customer written by name, matches one ERP customer':
        dict(unique_match(orders.loc[p_findings['name_reference'], 'Cliente'], ERP_BY_NAME)),
    'orders without product code, description matches one ERP product':
        dict(unique_match(orders.loc[missing_references(p_data, 'Produto'), 'Descrição'], ERP_BY_DESCRIPTION)),
    'receivables without customer, order has a single customer in orders sheet':
        f"{sum(1 for n in no_customer if n in order_customers.index and order_customers[n] == 1)} of {len(no_customer)}",
    'customers without code, company name matches one ERP customer':
        dict(unique_match(customers.loc[c_findings['missing_reference'], 'Razão Social'], ERP_BY_NAME)),
    'orders invalid quantity by shape':
        dict(Counter(map(quantity_shape, orders.loc[p_findings['invalid_quantity'], 'Qtd']))),
    'receivables reversed rows that have a negative amount':
        f"{int(receivables.loc[t_findings['reversed_receivable'], 'valor'].str.strip().str.startswith('-').sum())}"
        f" of {len(t_findings['reversed_receivable'])} (negative amounts in file: {t_negative})",
    'cancelled orders (distinct) that have receivables':
        f"{len(cancelled_numbers & set(receivables['num_pedido']))} of {len(cancelled_numbers)}",
    'customer codes repeated in the customers sheet':
        sorted(set(customers.loc[c_findings['key_duplicate'], 'Cód.'])),
}
evidence

## Problem report (count, code-only example, proposed rule, reconciliation)

In [ ]:
reports = [
    build_file_report(CUSTOMERS, customers, c_findings, ['Cód.'], HEADER_ROW[CUSTOMERS]),
    build_file_report(ORDERS, orders, p_findings, ['Nº Pedido', 'Produto'], HEADER_ROW[ORDERS]),
    build_file_report(RECEIVABLES, receivables, t_findings, ['id_titulo', 'parcela'], HEADER_ROW[RECEIVABLES]),
]
for report in reports:
    assert report.rows_read == report.rows_clean + report.rows_with_problems
problem_tables = render_markdown(reports)
print(problem_tables)

# Seller nicknames affect every row; show the reconciliation without them as well.
def without_sellers(findings):
    return {k: v for k, v in findings.items() if k != 'unmatched_seller'}

mapped = [
    build_file_report(CUSTOMERS, customers, without_sellers(c_findings), ['Cód.'], HEADER_ROW[CUSTOMERS]),
    build_file_report(ORDERS, orders, without_sellers(p_findings), ['Nº Pedido', 'Produto'], HEADER_ROW[ORDERS]),
]
mapped_lines = [f'{r.file_name}: {r.rows_read} = {r.rows_clean} without problems + '
                f'{r.rows_with_problems} with problems' for r in mapped]
print('\n'.join(mapped_lines))

## Write the tables used by the report (ignored by Git)

In [ ]:
def table(rows, headers):
    lines = ['| ' + ' | '.join(headers) + ' |', '| ' + ' | '.join(['---'] * len(headers)) + ' |']
    lines += ['| ' + ' | '.join(str(cell) for cell in row) + ' |' for row in rows]
    return '\n'.join(lines)

sections = [f'ERP export: {erp_exported_at:%Y-%m-%d %H:%M}', '']
for name, formats, variants in ((CUSTOMERS, c_formats, c_variants), (ORDERS, p_formats, p_variants),
                                (RECEIVABLES, t_formats, t_variants)):
    s = structure[name]
    sections += [f'## {name}', f'Rows: {s["rows"]}, columns: {s["columns"]}, header row: {HEADER_ROW[name]}', '',
                 table(s['types'].items(), ['Column', 'Type']), '',
                 table([(col, ', '.join(f'{k}={v}' for k, v in counts.items())) for col, counts in formats.items()],
                       ['Column', 'Formats']), '',
                 table([(col, norm, len(written), ', '.join(sorted(repr(w) for w in written)))
                        for col, groups in variants.items() for norm, written in groups.items()],
                       ['Column', 'Normalized', 'Variants', 'Written as']), '']
sections += ['## Extra observations',
             f'Orders Total column filled: {p_total_filled} of {len(orders)}',
             f'Receivables with negative amount: {t_negative}', '',
             table(p_invalid_values.head(15).items(), ['Invalid quantity', 'Rows']), '',
             '## ERP cross-check', table(((k, v) for k, v in cross_check.items()), ['Check', 'Result']), '',
             '## Evidence', table(evidence.items(), ['Check', 'Result']), '',
             '## Reconciliation if seller nicknames are mapped', *mapped_lines, '',
             '## Problems', problem_tables]
output = '\n'.join(sections)
(RAW / 'profile-tables.md').write_text(output, encoding='utf-8')
print('written', RAW / 'profile-tables.md')